    ## 02_merge_diploma

يدمج عمودي الدبلوم (diploma_gpa, diploma_type_id) على الجدول المدموج الكامل.

المصدر  : features/merged_add_acd_crg.parquet  (مخرج 01_merge)
الدبلوم : preprocessed/V_add_academic_info/v_add_adcademic_info_cleaned.parquet
المخرج  : preprocessed/merge/merged_with_diploma.parquet  (ملف جديد مميز — لا يعدّل شيئًا بالمكان)

student_id مطبّع بالمصدر (Read.ipynb) بـ normalize_id_columns — الطرفان string هنا.
diploma_type_id يُعاد لـ Int64 بعد الـ left join (pandas يرفعه لـ float بسبب NaN).

In [ ]:
import pandas as pd
from src.paths import  PREPROCESSED_DIR, MERGE_DIR

SOURCE_MERGED_PATH = MERGE_DIR / "merged_add_acd_crg.parquet"
DIPLOMA_SOURCE_PATH = PREPROCESSED_DIR / "V_add_academic_info" / "v_add_adcademic_info_cleaned.parquet"
OUTPUT_PATH = MERGE_DIR / "merged_with_diploma.parquet"

for p in [SOURCE_MERGED_PATH, DIPLOMA_SOURCE_PATH]:
    assert p.exists(), f"ملف مصدر مفقود: {p}"

print("مصدر المدموج :", SOURCE_MERGED_PATH)
print("مصدر الدبلوم :", DIPLOMA_SOURCE_PATH)
print("المخرج       :", OUTPUT_PATH)

In [ ]:
diploma_df = pd.read_parquet(DIPLOMA_SOURCE_PATH)
diploma_df.columns = diploma_df.columns.str.lower()

print("شكل مصدر الدبلوم :", diploma_df.shape)
print("الأعمدة          :", list(diploma_df.columns))
print("\nالأنواع:")
print(diploma_df.dtypes.to_string())

In [ ]:
_required = ['student_id', 'diploma_gpa', 'diploma_type_id']
_missing = [c for c in _required if c not in diploma_df.columns]
assert not _missing, f"أعمدة مطلوبة مفقودة من مصدر الدبلوم: {_missing}"

diploma_df = diploma_df[_required].copy()

_nulls = diploma_df.isna().sum()
assert _nulls.sum() == 0, f"مصدر الدبلوم فيه null بعد الاختيار:\n{_nulls.to_string()}"
print(f"أعمدة الدبلوم جاهزة: {_required}  ({len(diploma_df):,} صف، بلا null)")

In [ ]:
# student_id مفروض مطبّع لـ string بالمصدر (Read.ipynb). نتأكد هنا صراحة.
assert diploma_df['student_id'].dtype == 'string', (
    f"student_id بمصدر الدبلوم لازم يكون string (مطبّع بالمصدر)، بس هو "
    f"{diploma_df['student_id'].dtype}. شغّل Read.ipynb المعدّل أولًا."
)

_dup = int(diploma_df['student_id'].duplicated(keep=False).sum())
assert _dup == 0, (
    f"مصدر الدبلوم فيه {_dup} صف بـ student_id مكرّر. "
    "لازم يكون فريد قبل الدمج (many_to_one)."
)
print(f"student_id: string وفريد ({diploma_df['student_id'].nunique():,} طالب). PASSED")
print("عينة:", diploma_df['student_id'].head(3).tolist())

In [ ]:
# كل diploma_type_id لازم لاحقته النقطية موحّدة (نفس الجامعة). لو أكثر من لاحقة → أوقف.
_id_str = diploma_df['diploma_type_id'].astype('string')
_suffixes = set(_id_str.str.extract(r'\.([^.]+)$', expand=False).dropna().unique())

print("اللواحق الموجودة بـ diploma_type_id:", _suffixes)
assert len(_suffixes) <= 1, (
    f"diploma_type_id فيه أكثر من لاحقة جامعة: {_suffixes}. "
    "كل لاحقة = جامعة مختلفة؛ منطق الدمج/الـ bucketing لازم يُراجع أولًا."
)

In [ ]:
if len(_suffixes) == 1:
    _s = next(iter(_suffixes))
    print(f"اللاحقة '.{_s}' موحّدة — تجريدها و cast لـ Int64.")
    diploma_df['diploma_type_id'] = (
        _id_str.str.split('.').str[0]
        .pipe(pd.to_numeric, errors='coerce')
        .astype('Int64')
    )
else:
    print("لا لاحقة — cast مباشر لـ Int64.")
    diploma_df['diploma_type_id'] = (
        pd.to_numeric(diploma_df['diploma_type_id'], errors='coerce').round().astype('Int64')
    )

print("نوع diploma_type_id بعد التجريد:", diploma_df['diploma_type_id'].dtype)
print(diploma_df['diploma_type_id'].value_counts(dropna=False).to_string())

In [ ]:
df = pd.read_parquet(SOURCE_MERGED_PATH)
print("شكل المدموج:", df.shape)

# student_id بالطرفين لازم نفس النوع (string) عشان الدمج يطابق صح.
assert df['student_id'].dtype == 'string', (
    f"student_id بالمدموج لازم string، بس هو {df['student_id'].dtype}."
)
print("student_id بالمدموج: string. عينة:", df['student_id'].head(3).tolist())

# الحارس ضد التطبيق المزدوج: لو أعمدة الدبلوم موجودة أصلًا → أوقف.
_existing = [c for c in ['diploma_gpa', 'diploma_type_id'] if c in df.columns]
assert not _existing, (
    f"أعمدة الدبلوم {_existing} موجودة أصلًا بالمدموج. "
    "هذا يعني الدمج انطبّق مرّتين — أوقف وحقّق."
)

In [ ]:
_rows_before = len(df)

df_merged = pd.merge(
    df,
    diploma_df[['student_id', 'diploma_gpa', 'diploma_type_id']],
    on='student_id',
    how='left',
    validate='many_to_one',   # يرفع خطأ لو diploma_df فيه student_id مكرّر
)

_rows_after = len(df_merged)
assert _rows_after == _rows_before, (
    f"عدد الصفوف تغيّر بعد left join — ممنوع يصير!\n"
    f"  قبل : {_rows_before:,}\n  بعد  : {_rows_after:,}\n"
    "غالبًا عدم تطابق نوع student_id بين الطرفين."
)
print(f"الدمج تم. الصفوف ثابتة: {_rows_after:,}")

In [ ]:
_matched = int(df_merged['diploma_type_id'].notna().sum())
_unmatched = int(df_merged['diploma_type_id'].isna().sum())
_unmatched_students = int(
    df_merged.loc[df_merged['diploma_type_id'].isna(), 'student_id'].nunique()
)

print(f"صفوف متطابقة    : {_matched:,}")
print(f"صفوف غير متطابقة : {_unmatched:,}  ({_unmatched / _rows_after * 100:.3f}%)")
print(f"طلاب بلا دبلوم   : {_unmatched_students:,}")
print("\nنكرّر: غير المتطابق متوقّع لطلاب قدامى بلا سجل دبلوم (فحص المرحلة 3: ~48 صف / 6 طلاب).")
print("\ndiploma_type_id بعد الدمج (dropna=False):")
print(df_merged['diploma_type_id'].value_counts(dropna=False).to_string())

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_merged.to_parquet(OUTPUT_PATH, index=False)

_mb = OUTPUT_PATH.stat().st_size / 1_048_576
print(f"انحفظ: {OUTPUT_PATH}")
print(f"الشكل النهائي: {df_merged.shape}")
print(f"أعمدة جديدة: diploma_gpa, diploma_type_id")
print(f"الحجم: {_mb:.1f} MB")
print("\nالتالي: select يقرأ من هذا الملف.")